# Notebook 2 – Ghép 2 ảnh bằng SIFT + RANSAC + Homography

1. Tạo 2 ảnh có phần chồng lấn từ `images/img.jpg` (ảnh 2 bị xoay 15°).
2. SIFT cho mỗi ảnh → so khớp descriptor (láng giềng gần nhất + ratio test).
3. RANSAC → ma trận Homography H (3×3).
4. Uốn ảnh 2 về hệ tọa độ ảnh 1 (`ops.warp_perspective` tự viết), chồng 2 ảnh, cắt viền đen.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("python"))   # notebook đặt ở thư mục gốc repo
import numpy as np
import matplotlib.pyplot as plt
import image_ops as ops
from sift_extractor import SIFT
from panorama_stitching import Panorama
from run_panorama import read_rgb              # chỉ dùng OpenCV để ĐỌC file ảnh
from general_functions import show_images, normalize_img
%matplotlib inline

In [ ]:
img = read_rgb("images/img.jpg") / 255.0
img1 = img[:, :800]
img2 = ops.rotate(img[:, 200:], 15)           # ảnh 2 = phần bên phải, xoay 15 độ
show_images([img1, img2], ["Ảnh 1", "Ảnh 2"], figsize=(12, 4), show_axis=False)

In [ ]:
sift = SIFT(num_downsamples=0)
f1 = sift.extract_sift_data(ops.rgb2gray(img1))
f2 = sift.extract_sift_data(ops.rgb2gray(img2))
print("Số keypoint:", len(f1), len(f2))

## So khớp + RANSAC
`compute_homography_matrix(src, dst)` trả về H biến tọa độ ảnh **nguồn** sang ảnh **đích**. Ở đây ta đưa ảnh 2 về hệ tọa độ ảnh 1.

In [ ]:
pano = Panorama()
H = pano.compute_homography_matrix(f2, f1)
print("Kết quả so khớp:", pano.match_info[-1])
print("Ma trận Homography H =\n", np.round(H, 4))

In [ ]:
h1, w1 = img1.shape[:2]; h2, w2 = img2.shape[:2]
img2_warped = ops.warp_perspective(img2, H, (w1 + w2, h1))
show_images([img1, img2_warped], ["Ảnh 1", "Ảnh 2 sau khi uốn theo H"], figsize=(14, 4))

In [ ]:
stitched = pano.stitch_two_images(img1, img2_warped)
x, y, w, h = pano.find_bounding_box(stitched)
show_images([stitched, stitched[y:y + h, x:x + w]], ["Sau khi ghép", "Sau khi cắt viền đen"], figsize=(14, 5), show_axis=False)